In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import chi2_contingency

df = pd.read_excel('Research Survey.xlsx')

# Create BNPL flag
df['is_bnpl'] = df['Q4_Used_BNPL'].str.contains('Yes')

# Encode Likert scales to numbers
likert_map = {'Strongly Disagree': 1, 'Disagree': 2, 'Neutral': 3, 'Agree': 4, 'Strongly Agree': 5}
freq_map = {'Very Low': 1, 'Low': 2, 'Moderate': 3, 'High': 4, 'Very High': 5}
spend_map = {'Very Low': 1, 'Low': 2, 'Moderate': 3, 'High': 4, 'Very High': 5}

df['shop_freq_num'] = df['Q5a_Shopping_Freq'].map(freq_map)
df['spend_num'] = df['Q5b_Typical_Spend'].map(spend_map)
df['price_imp_num'] = df['Q5c_Price_Importance'].map(freq_map)

# Encode BNPL behavior items (only for BNPL group)
for col in ['Q8a_Increases_Items','Q8b_More_Expensive_Brands','Q8c_Less_Worried','Q8d_Shop_More_Freq','Q8e_Manage_Cash_Flow']:
    df[col+'_num'] = df[col].map(likert_map)

bnpl = df[df['is_bnpl']==True]
non_bnpl = df[df['is_bnpl']==False]

print("=== T-TEST: Shopping Frequency ===")
t, p = stats.ttest_ind(bnpl['shop_freq_num'].dropna(), non_bnpl['shop_freq_num'].dropna())
print(f"BNPL mean: {bnpl['shop_freq_num'].mean():.2f}, Non-BNPL mean: {non_bnpl['shop_freq_num'].mean():.2f}")
print(f"t={t:.3f}, p={p:.4f}")

print("\n=== T-TEST: Typical Spend ===")
t, p = stats.ttest_ind(bnpl['spend_num'].dropna(), non_bnpl['spend_num'].dropna())
print(f"BNPL mean: {bnpl['spend_num'].mean():.2f}, Non-BNPL mean: {non_bnpl['spend_num'].mean():.2f}")
print(f"t={t:.3f}, p={p:.4f}")

print("\n=== T-TEST: Price Importance ===")
t, p = stats.ttest_ind(bnpl['price_imp_num'].dropna(), non_bnpl['price_imp_num'].dropna())
print(f"BNPL mean: {bnpl['price_imp_num'].mean():.2f}, Non-BNPL mean: {non_bnpl['price_imp_num'].mean():.2f}")
print(f"t={t:.3f}, p={p:.4f}")

print("\n=== BNPL group: Behavior items means ===")
for col in ['Q8a_Increases_Items','Q8b_More_Expensive_Brands','Q8c_Less_Worried','Q8d_Shop_More_Freq','Q8e_Manage_Cash_Flow']:
    print(f"{col}: {bnpl[col+'_num'].mean():.2f}")

# Monthly spending influence breakdown for BNPL group
print("\n=== Monthly Spending Influence (BNPL group) ===")
print(bnpl['Q9_Monthly_Spending_Influence'].value_counts())

# Recommend scale
print("\n=== Recommend Scale (BNPL group) ===")
print(f"Mean: {bnpl['Q10_Recommend_Scale_1_10'].mean():.2f}, Median: {bnpl['Q10_Recommend_Scale_1_10'].median():.1f}")

# Chi-square: BNPL use vs monthly spending influence
# Filled NA values to prevent crashes if non-BNPL users skipped this question
ct = pd.crosstab(df['is_bnpl'], df['Q9_Monthly_Spending_Influence'].fillna('No Data'))
print("\n=== Chi-square: BNPL vs Monthly Spending Influence ===")
print(ct)

try:
    chi2, p_chi, dof, expected = chi2_contingency(ct)
    print(f"Chi2={chi2:.3f}, p={p_chi:.4f}, dof={dof}")
except ValueError as e:
    print(f"Could not run chi-square (likely due to missing non-user data): {e}")

=== T-TEST: Shopping Frequency ===
BNPL mean: 3.06, Non-BNPL mean: 2.90
t=0.848, p=0.3970

=== T-TEST: Typical Spend ===
BNPL mean: 2.98, Non-BNPL mean: 3.22
t=-1.339, p=0.1819

=== T-TEST: Price Importance ===
BNPL mean: 2.94, Non-BNPL mean: 3.06
t=-0.682, p=0.4960

=== BNPL group: Behavior items means ===
Q8a_Increases_Items: 3.60
Q8b_More_Expensive_Brands: 3.55
Q8c_Less_Worried: 3.34
Q8d_Shop_More_Freq: 3.54
Q8e_Manage_Cash_Flow: 3.70

=== Monthly Spending Influence (BNPL group) ===
Q9_Monthly_Spending_Influence
My total spending has slightly increased         58
My total spending has significantly increased    37
My total spending has remained the same          19
My total spending has significantly decreased     7
My total spending has slightly decreased          4
Name: count, dtype: int64

=== Recommend Scale (BNPL group) ===
Mean: 7.54, Median: 8.0

=== Chi-square: BNPL vs Monthly Spending Influence ===
Q9_Monthly_Spending_Influence  My total spending has remained the same  \
i